# 06 SFT Walkthrough

SFT means **Supervised Fine-Tuning**.

In nanochat, SFT takes the pretrained base model and trains it on conversations/tasks so it becomes useful as a chat model:

```text
base checkpoint -> chat_sft.py -> chatsft checkpoint
raw next-token model -> assistant-style model
```

This notebook connects small examples to the real SFT path in nanochat:

```text
Task objects -> conversation dictionaries -> tokenizer.render_conversation(...)
ids + assistant mask -> inputs + targets -> model(x, y) -> chatsft_checkpoints/
```

It uses your real `nanochat-artifacts/` folder when available, but avoids downloading Hugging Face datasets or loading the 3.9GB model weights.

In [1]:
from pathlib import Path
import json
import os
import re
import sys

repo_root = Path.cwd()
if not ((repo_root / "nanochat").exists() and (repo_root / "pyproject.toml").exists()):
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "nanochat").exists() and (candidate / "pyproject.toml").exists():
            repo_root = candidate
            break

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

missing = []
for package in ["torch", "rustbpe", "tiktoken", "tokenizers"]:
    try:
        __import__(package)
    except Exception:
        missing.append(package)

if missing:
    raise RuntimeError(
        "Missing dependencies: " + ", ".join(missing) + "\n"
        "From the repo root run:\n"
        "  uv sync --extra cpu --group dev\n"
        "or, on CUDA machines:\n"
        "  uv sync --extra gpu --group dev\n"
        "Then reopen this notebook with the nanochat (.venv) kernel."
    )

import torch

from nanochat.common import get_base_dir
from nanochat.tokenizer import RustBPETokenizer

artifact_root = Path(os.environ.get("NANOCHAT_ARTIFACTS_DIR", repo_root / "nanochat-artifacts")).expanduser()

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def read_text(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()

def truncate(text, n=100):
    text = str(text).replace("\n", "\\n")
    return text if len(text) <= n else text[: n - 3] + "..."

def print_kv(rows, key_width=26):
    for key, value in rows:
        print(f"{key:<{key_width}} {value}")

def find_tokenizer_dir():
    candidates = [
        artifact_root / "tokenizer",
        Path(get_base_dir()) / "tokenizer",
    ]
    for candidate in candidates:
        if (candidate / "tokenizer.pkl").exists():
            return candidate
    return None

tokenizer_dir = find_tokenizer_dir()
tokenizer = RustBPETokenizer.from_directory(tokenizer_dir) if tokenizer_dir else None

print(f"repo root:      {repo_root}")
print(f"artifact root:  {artifact_root} (exists={artifact_root.exists()})")
print(f"base dir:       {get_base_dir()}")
print(f"tokenizer dir:  {tokenizer_dir}")

repo root:      /Users/eugene/Developer/nanochat
artifact root:  /Users/eugene/Developer/nanochat/nanochat-artifacts (exists=True)
base dir:       /Users/eugene/.cache/nanochat
tokenizer dir:  /Users/eugene/Developer/nanochat/nanochat-artifacts/tokenizer


## Step 1. SFT Starts From The Base Model

The real SFT script is `scripts/chat_sft.py`.

The important first move is that it does **not** initialize a new random model. It loads the pretrained base checkpoint:

```python
model, tokenizer, meta = load_model("base", device, phase="train", ...)
```

Then SFT continues training that model on chat/task examples and saves the result under `chatsft_checkpoints/`.

So mentally:

```text
base_checkpoints/d24/model_005568.pt
        -> SFT training
chatsft_checkpoints/d24/model_000483.pt
```

In [2]:
base_meta_path = artifact_root / "base_checkpoints" / "d24" / "meta_005568.json"
sft_meta_path = artifact_root / "chatsft_checkpoints" / "d24" / "meta_000483.json"

if base_meta_path.exists() and sft_meta_path.exists():
    base_meta = read_json(base_meta_path)
    sft_meta = read_json(sft_meta_path)
    print("Real checkpoint transition from your artifacts:")
    print_kv([
        ("base step", base_meta["step"]),
        ("base val_bpb", f"{base_meta['val_bpb']:.4f}"),
        ("SFT step", sft_meta["step"]),
        ("SFT val_bpb", f"{sft_meta['val_bpb']:.4f}"),
        ("same n_layer", base_meta["model_config"]["n_layer"] == sft_meta["model_config"]["n_layer"]),
        ("same vocab_size", base_meta["model_config"]["vocab_size"] == sft_meta["model_config"]["vocab_size"]),
        ("same sequence_len", base_meta["model_config"]["sequence_len"] == sft_meta["model_config"]["sequence_len"]),
    ])
else:
    print("No local RunPod artifacts found. This notebook will still use toy examples.")

Real checkpoint transition from your artifacts:
base step                  5568
base val_bpb               0.7182
SFT step                   483
SFT val_bpb                0.2734
same n_layer               True
same vocab_size            True
same sequence_len          True


## Step 2. SFT Data Is A Mixture Of Conversation Tasks

Base training reads long text streams from parquet files.

SFT is different: each training item is a **conversation dictionary** with messages:

```python
{
    "messages": [
        {"role": "user", "content": "..."},
        {"role": "assistant", "content": "..."},
    ]
}
```

A **Task** is nanochat's lightweight dataset wrapper. Each task returns conversations in that same shape.

For example:

- `SmolTalk` is a Task that loads general chat conversations from Hugging Face.
- `MMLU` is a Task that turns multiple-choice questions into user/assistant conversations.
- `GSM8K` is a Task that turns math problems into user/assistant conversations.
- `CustomJSON` is a Task that reads your own `.jsonl` file of conversations.
- `SimpleSpelling` and `SpellingBee` are Tasks that generate spelling/counting conversations.

A **TaskMixture** is just a wrapper that combines multiple Tasks into one bigger training dataset.

So this real SFT line:

```python
train_dataset = TaskMixture(train_tasks)
```

means:

```text
combine SmolTalk + identity conversations + MMLU + GSM8K + spelling tasks
then feed the mixed conversations to SFT training
```

That mixture is why the SFT model is not just more fluent. It is being pushed toward chat format, task format, and specific assistant behaviors.

The next cell maps those names to the actual nanochat files and lines that use them.

In [3]:
actual_sft_usage = [
    ("TaskMixture", "tasks/common.py", "scripts/chat_sft.py:174,176", "combines task datasets into train/val datasets"),
    ("SmolTalk", "tasks/smoltalk.py", "scripts/chat_sft.py:166,177", "general chat conversations from Hugging Face"),
    ("CustomJSON", "tasks/customjson.py", "scripts/chat_sft.py:167,168", "identity_conversations.jsonl, intentionally included twice"),
    ("MMLU", "tasks/mmlu.py", "scripts/chat_sft.py:169,178", "multiple-choice conversations"),
    ("GSM8K", "tasks/gsm8k.py", "scripts/chat_sft.py:170,179", "math/tool-use conversations"),
    ("SimpleSpelling", "tasks/spellingbee.py", "scripts/chat_sft.py:171", "spelling conversations"),
    ("SpellingBee", "tasks/spellingbee.py", "scripts/chat_sft.py:172", "letter-counting conversations"),
]

print("Actual classes used by SFT training:")
print(f"{'class':<16} {'defined in':<24} {'used in':<28} purpose")
print("-" * 100)
for cls, defined_in, used_in, purpose in actual_sft_usage:
    print(f"{cls:<16} {defined_in:<24} {used_in:<28} {purpose}")

sft_log = artifact_root / "sft_retry.log"
if sft_log.exists():
    for line in read_text(sft_log).splitlines():
        if "Training mixture:" in line:
            print("\nFrom your actual SFT retry log:")
            print(line)
            break

Actual classes used by SFT training:
class            defined in               used in                      purpose
----------------------------------------------------------------------------------------------------
TaskMixture      tasks/common.py          scripts/chat_sft.py:174,176  combines task datasets into train/val datasets
SmolTalk         tasks/smoltalk.py        scripts/chat_sft.py:166,177  general chat conversations from Hugging Face
CustomJSON       tasks/customjson.py      scripts/chat_sft.py:167,168  identity_conversations.jsonl, intentionally included twice
MMLU             tasks/mmlu.py            scripts/chat_sft.py:169,178  multiple-choice conversations
GSM8K            tasks/gsm8k.py           scripts/chat_sft.py:170,179  math/tool-use conversations
SimpleSpelling   tasks/spellingbee.py     scripts/chat_sft.py:171      spelling conversations
SpellingBee      tasks/spellingbee.py     scripts/chat_sft.py:172      letter-counting conversations

From your actual SFT re

### What SFT Downloads

SFT does not download another huge pretraining corpus. It downloads a few task datasets and then mixes them.

Approximate sizes below are **download/cache size**, not the number of rows seen during training. Oversampling MMLU/GSM8K/CustomJSON repeats examples in the mixture, but it does not download extra copies.

| Task | Actual source | Approx download | How nanochat uses it |
|---|---:|---:|---|
| `SmolTalk` | [`HuggingFaceTB/smol-smoltalk`](https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk) | ~971 MB | General user/assistant chat conversations. |
| `CustomJSON` | local `identity_conversations.jsonl` | ~2.4 MB in your artifacts | Identity/personality conversations, included twice. |
| `MMLU` | [`cais/mmlu`](https://huggingface.co/datasets/cais/mmlu) | ~51 MB | Multiple-choice question -> answer letter. |
| `GSM8K` | [`openai/gsm8k`](https://huggingface.co/datasets/openai/gsm8k) | ~2.7 MB | Math problem -> solution, with tool-call parts parsed from `<<expr=result>>`. |
| `SimpleSpelling` | [`words_alpha.txt`](https://github.com/dwyl/english-words/blob/master/words_alpha.txt) | ~4 MB | Generated spelling conversations from the word list. |
| `SpellingBee` | same `words_alpha.txt` file | already downloaded above | Generated letter-counting conversations from the same word list. |

So the practical mental model is:

```text
SmolTalk is the big SFT download.
MMLU/GSM8K are small.
SimpleSpelling and SpellingBee share one tiny word-list download.
CustomJSON is your local identity file.
```

In [4]:
sft_sources = [
    ("SmolTalk", "HuggingFaceTB/smol-smoltalk", "~971 MB", "tasks/smoltalk.py:16"),
    ("CustomJSON", str(artifact_root / "identity_conversations.jsonl"), "local file", "tasks/customjson.py"),
    ("MMLU", "cais/mmlu", "~51 MB", "tasks/mmlu.py:20"),
    ("GSM8K", "openai/gsm8k", "~2.7 MB", "tasks/gsm8k.py:43"),
    ("SimpleSpelling", "dwyl/english-words words_alpha.txt", "~4 MB", "tasks/spellingbee.py:37,241"),
    ("SpellingBee", "same words_alpha.txt", "shared", "tasks/spellingbee.py:37,122"),
]

print(f"{'task':<16} {'source':<48} {'approx':<10} actual code")
print("-" * 105)
for task, source, approx, code_ref in sft_sources:
    print(f"{task:<16} {truncate(source, 46):<48} {approx:<10} {code_ref}")

identity_path = artifact_root / "identity_conversations.jsonl"
if identity_path.exists():
    print()
    print("Your local CustomJSON file:")
    print_kv([
        ("path", identity_path),
        ("size", f"{identity_path.stat().st_size / 1024 / 1024:.1f} MB"),
        ("lines", f"{len(identity_path.read_text(encoding='utf-8').splitlines()):,}"),
    ])


task             source                                           approx     actual code
---------------------------------------------------------------------------------------------------------
SmolTalk         HuggingFaceTB/smol-smoltalk                      ~971 MB    tasks/smoltalk.py:16
CustomJSON       /Users/eugene/Developer/nanochat/nanochat-a...   local file tasks/customjson.py
MMLU             cais/mmlu                                        ~51 MB     tasks/mmlu.py:20
GSM8K            openai/gsm8k                                     ~2.7 MB    tasks/gsm8k.py:43
SimpleSpelling   dwyl/english-words words_alpha.txt               ~4 MB      tasks/spellingbee.py:37,241
SpellingBee      same words_alpha.txt                             shared     tasks/spellingbee.py:37,122

Your local CustomJSON file:
path                       /Users/eugene/Developer/nanochat/nanochat-artifacts/identity_conversations.jsonl
size                       2.4 MB
lines                      1,000


### Example Conversation Chunks From Each Task

The examples below are small on purpose, but each one matches the shape returned by the real task's `get_example(...)` method.

This is the useful thing to notice: after task-specific loading/parsing, everything becomes the same kind of conversation object before it reaches `tokenizer.render_conversation(...)`.

In [5]:
def print_conversation_example(name, conversation):
    print("=" * 90)
    print(name)
    print("=" * 90)
    for message in conversation["messages"]:
        role = message["role"]
        content = message["content"]
        print(f"{role}:")
        if isinstance(content, list):
            for part in content:
                print(f"  [{part['type']}] {truncate(part['text'], 140)}")
        else:
            print(f"  {truncate(content, 180)}")
    print()

examples = []

# CustomJSON: if your artifact exists, show real local data. Otherwise show the expected JSONL shape.
identity_path = artifact_root / "identity_conversations.jsonl"
if identity_path.exists():
    first_identity_messages = json.loads(identity_path.read_text(encoding="utf-8").splitlines()[0])
    examples.append(("CustomJSON from identity_conversations.jsonl", {"messages": first_identity_messages}))
else:
    examples.append(("CustomJSON expected shape", {
        "messages": [
            {"role": "user", "content": "Who are you?"},
            {"role": "assistant", "content": "I am nanochat, a small chat model trained in this repo."},
        ]
    }))

# SmolTalk: tasks/smoltalk.py returns row['messages'] directly.
examples.append(("SmolTalk row['messages'] shape", {
    "messages": [
        {"role": "user", "content": "Can you give me a quick dinner idea?"},
        {"role": "assistant", "content": "Try rice bowls with roasted vegetables, a protein, and a simple yogurt sauce."},
    ]
}))

# MMLU: tasks/mmlu.py renders choices into the user message and the assistant is one letter.
examples.append(("MMLU multiple-choice conversation shape", {
    "messages": [
        {"role": "user", "content": "What is the capital of France?\n\nA. Berlin\nB. Madrid\nC. Paris\nD. Rome"},
        {"role": "assistant", "content": "C"},
    ],
    "letters": ("A", "B", "C", "D"),
}))

# GSM8K: tasks/gsm8k.py turns <<expr=result>> spans into python and python_output parts.
examples.append(("GSM8K math/tool-use conversation shape", {
    "messages": [
        {"role": "user", "content": "A box has 3 red balls and 4 blue balls. How many balls are there?"},
        {"role": "assistant", "content": [
            {"type": "text", "text": "There are 3 red balls and 4 blue balls, so compute "},
            {"type": "python", "text": "3+4"},
            {"type": "python_output", "text": "7"},
            {"type": "text", "text": ".\n#### 7"},
        ]},
    ]
}))

# SimpleSpelling and SpellingBee both come from words_alpha.txt, but generate different conversations.
examples.append(("SimpleSpelling generated from words_alpha.txt", {
    "messages": [
        {"role": "user", "content": "Spell the word: apple"},
        {"role": "assistant", "content": "apple:a,p,p,l,e"},
    ]
}))

examples.append(("SpellingBee generated from the same words_alpha.txt", {
    "messages": [
        {"role": "user", "content": "How many p are in apple?"},
        {"role": "assistant", "content": [
            {"type": "text", "text": "First spell the word out:\napple:a,p,p,l,e\nThen count p: two hits."},
            {"type": "python", "text": "'apple'.count('p')"},
            {"type": "python_output", "text": "2"},
            {"type": "text", "text": "\n\nMy final answer is:\n\n#### 2"},
        ]},
    ]
}))

for name, conversation in examples:
    print_conversation_example(name, conversation)


CustomJSON from identity_conversations.jsonl
user:
  Hey nanochat! I'm a CS student studying transformers and I'm really curious about how long it actually takes to train a model like you from zero. Is it a weeks-long process?
assistant:
  That's a great question! Actually, it's much faster than you might think. While original models like GPT-2 took about a week to train back in 2019, I can be trained from scratch...
user:
  Only 3 hours? That sounds crazy fast for a 1.4B parameter model. What kind of hardware are we talking about to hit those speeds, and does it cost a fortune?
assistant:
  It is quite efficient! To hit that 3-hour mark, I require a single node with 8xH100 GPUs. Because the training is so optimized, the total cost comes out to approximately $73 (at...

SmolTalk row['messages'] shape
user:
  Can you give me a quick dinner idea?
assistant:
  Try rice bowls with roasted vegetables, a protein, and a simple yogurt sauce.

MMLU multiple-choice conversation shape
user:
  Wha

### Optional: Preview Real Hugging Face Rows

The next cell is disabled by default so `Run All` does not accidentally download ~1GB of SmolTalk.

If you want to inspect real remote rows, change `RUN_LIVE_DATASET_PREVIEW = False` to `True`.

In [6]:
RUN_LIVE_DATASET_PREVIEW = False

if RUN_LIVE_DATASET_PREVIEW:
    from datasets import load_dataset

    previews = [
        ("SmolTalk", load_dataset("HuggingFaceTB/smol-smoltalk", split="train", streaming=True)),
        ("MMLU", load_dataset("cais/mmlu", "all", split="auxiliary_train", streaming=True)),
        ("GSM8K", load_dataset("openai/gsm8k", "main", split="train", streaming=True)),
    ]

    for name, ds in previews:
        row = next(iter(ds))
        print("=" * 90)
        print(name)
        print(row)
else:
    print("Live dataset preview skipped. Set RUN_LIVE_DATASET_PREVIEW = True to download/stream real rows.")


Live dataset preview skipped. Set RUN_LIVE_DATASET_PREVIEW = True to download/stream real rows.


### What each actual Task returns

The important unifying point is that every Task has a `get_example(...)` method that returns a conversation dictionary.

The implementations differ in where the data comes from, but they all end up shaped for `tokenizer.render_conversation(...)`.

The next cell is not a separate toy dataset. It is a compact map of the actual `get_example(...)` behavior in the task files.

In [7]:
actual_get_example_map = [
    ("SmolTalk", "tasks/smoltalk.py:16,23-45", "loads HuggingFaceTB/smol-smoltalk, returns row['messages']"),
    ("CustomJSON", "tasks/customjson.py:35-52,59-63", "reads each JSONL line into messages, returns {'messages': messages}"),
    ("MMLU", "tasks/mmlu.py:29-48", "renders question choices as user text, assistant is the correct letter"),
    ("GSM8K", "tasks/gsm8k.py:52-85", "renders math question, parses <<expr=result>> into python/tool-output parts"),
    ("SimpleSpelling", "tasks/spellingbee.py:233+", "generates spelling conversations"),
    ("SpellingBee", "tasks/spellingbee.py:115-205", "generates letter-counting conversations with Python verification parts"),
]

print(f"{'class':<16} {'actual code':<32} what get_example returns")
print("-" * 105)
for cls, code_ref, behavior in actual_get_example_map:
    print(f"{cls:<16} {code_ref:<32} {behavior}")

print("\nAfter this point, chat_sft.py treats all of these as conversations and sends them to tokenizer.render_conversation(...).")


class            actual code                      what get_example returns
---------------------------------------------------------------------------------------------------------
SmolTalk         tasks/smoltalk.py:16,23-45       loads HuggingFaceTB/smol-smoltalk, returns row['messages']
CustomJSON       tasks/customjson.py:35-52,59-63  reads each JSONL line into messages, returns {'messages': messages}
MMLU             tasks/mmlu.py:29-48              renders question choices as user text, assistant is the correct letter
GSM8K            tasks/gsm8k.py:52-85             renders math question, parses <<expr=result>> into python/tool-output parts
SimpleSpelling   tasks/spellingbee.py:233+        generates spelling conversations
SpellingBee      tasks/spellingbee.py:115-205     generates letter-counting conversations with Python verification parts

After this point, chat_sft.py treats all of these as conversations and sends them to tokenizer.render_conversation(...).


## Step 3. Oversampling Means Adding The Same Actual Task Twice

Now that `CustomJSON` means "read `identity_conversations.jsonl` as conversations" and `TaskMixture` means "combine task datasets", this real code should read more clearly:

```python
train_tasks = [
    SmolTalk(split="train"),
    CustomJSON(filepath=identity_conversations_filepath),
    CustomJSON(filepath=identity_conversations_filepath),
    ...
]
train_dataset = TaskMixture(train_tasks)
```

The thing added twice is the **same identity JSONL dataset**, not a new class and not a different file.

If `identity_conversations.jsonl` has 1,000 conversations, adding it once gives 1,000 identity training examples. Adding it twice gives 2,000 identity slots in the mixed SFT dataset.

This is called oversampling. nanochat does it because the identity dataset is tiny compared with SmolTalk, so without oversampling the model would barely see those identity/personality examples.


In [8]:
identity_path = artifact_root / "identity_conversations.jsonl"
identity_count = None
if identity_path.exists():
    identity_count = len(identity_path.read_text(encoding="utf-8").splitlines())

# This is the actual multiplicity from scripts/chat_sft.py lines 167-168.
customjson_copies_in_train_tasks = 2

print("Actual oversampling in chat_sft.py:")
print("  line 167: CustomJSON(filepath=identity_conversations_filepath)")
print("  line 168: CustomJSON(filepath=identity_conversations_filepath)")
print()
if identity_count is not None:
    print(f"identity_conversations.jsonl conversations: {identity_count:,}")
    print(f"CustomJSON copies in train_tasks:          {customjson_copies_in_train_tasks}")
    print(f"identity slots in TaskMixture:            {identity_count * customjson_copies_in_train_tasks:,}")
else:
    print(f"No local identity file found at {identity_path}")

if sft_log.exists():
    for line in read_text(sft_log).splitlines():
        if "Training mixture:" in line:
            print()
            print("Actual total mixed SFT dataset from your run:")
            print(" ", line)
            break

Actual oversampling in chat_sft.py:
  line 167: CustomJSON(filepath=identity_conversations_filepath)
  line 168: CustomJSON(filepath=identity_conversations_filepath)

identity_conversations.jsonl conversations: 1,000
CustomJSON copies in train_tasks:          2
identity slots in TaskMixture:            2,000

Actual total mixed SFT dataset from your run:
  Training mixture: 1,071,759 rows (MMLU x3, GSM8K x4)


## Step 4. Your Custom Identity Data

The speedrun downloads `identity_conversations.jsonl` before SFT.

Each line is one supervised conversation. `CustomJSON` reads those lines and returns the same conversation shape as SmolTalk/MMLU/GSM8K.

This is the easiest place to customize model personality/identity without changing architecture.

In [9]:
identity_path = artifact_root / "identity_conversations.jsonl"
if identity_path.exists():
    lines = identity_path.read_text(encoding="utf-8").splitlines()
    print(f"identity conversations: {len(lines):,}")
    print()
    for idx, line in enumerate(lines[:2]):
        messages = json.loads(line)
        print(f"conversation {idx}")
        for message in messages:
            print(f"  {message['role']:<9} {truncate(message['content'], 140)}")
        print()
else:
    print(f"No identity file found at {identity_path}")

identity conversations: 1,000

conversation 0
  user      Hey nanochat! I'm a CS student studying transformers and I'm really curious about how long it actually takes to train a model like you fr...
  assistant That's a great question! Actually, it's much faster than you might think. While original models like GPT-2 took about a week to train bac...
  user      Only 3 hours? That sounds crazy fast for a 1.4B parameter model. What kind of hardware are we talking about to hit those speeds, and does...
  assistant It is quite efficient! To hit that 3-hour mark, I require a single node with 8xH100 GPUs. Because the training is so optimized, the total...

conversation 1
  user      hey! are you like the same thing as chatgpt? how did you get so smart?
  assistant Hello! I am nanochat. While I can chat like ChatGPT, I am a bit different. I am an open-source project created by Andrej Karpathy in Janu...
  user      that sounds expensive. do you need like a supercomputer that costs millions of

## Step 5. `render_conversation` Adds Chat Special Tokens And A Loss Mask

This is the core idea of SFT in nanochat.

The tokenizer does not only return token ids. For chat training it returns:

```text
ids  = token ids for the rendered conversation
mask = 1 where the assistant should be trained, 0 everywhere else
```

User text is context, not something the model is punished for failing to predict. Assistant answer tokens are supervised.

So SFT teaches:

```text
given user prompt -> predict assistant response
```

not:

```text
memorize both user and assistant text equally
```

In [10]:
toy_conversation = {
    "messages": [
        {"role": "user", "content": "What is 2+2?"},
        {"role": "assistant", "content": "2+2 is 4."},
    ]
}

if tokenizer is None:
    print("No trained tokenizer found, so this cell cannot call render_conversation.")
else:
    ids, mask = tokenizer.render_conversation(toy_conversation)
    print(f"number of tokens: {len(ids)}")
    print()
    print(f"{'i':>2}  {'token id':>8}  {'mask':>4}  decoded")
    print("-" * 48)
    for i, (token_id, mask_value) in enumerate(zip(ids, mask)):
        decoded = tokenizer.decode([token_id]).replace("\n", "\\n")
        print(f"{i:>2}  {token_id:>8}  {mask_value:>4}  {decoded!r}")
    print()
    print("mask=0: context/special/user tokens")
    print("mask=1: assistant answer tokens that contribute to loss")

number of tokens: 19

 i  token id  mask  decoded
------------------------------------------------
 0     32759     0  '<|bos|>'
 1     32760     0  '<|user_start|>'
 2      1089     0  'What'
 3       306     0  ' is'
 4        32     0  ' '
 5        50     0  '2'
 6        43     0  '+'
 7        50     0  '2'
 8        63     0  '?'
 9     32761     0  '<|user_end|>'
10     32762     0  '<|assistant_start|>'
11        50     1  '2'
12        43     1  '+'
13        50     1  '2'
14       306     1  ' is'
15        32     1  ' '
16        52     1  '4'
17        46     1  '.'
18     32763     1  '<|assistant_end|>'

mask=0: context/special/user tokens
mask=1: assistant answer tokens that contribute to loss


## Step 6. Mask Becomes `targets == -1`

This is the same idea you saw in the training notebook, but SFT has an extra mask.

Real code in `chat_sft.py` does this:

```text
inputs  = row[:, :-1]
targets = row[:, 1:]
targets[mask[:, 1:] == 0] = -1
```

`-1` means ignore this target in cross-entropy.

So the model still **reads** user tokens as context, but loss is computed only on assistant-answer tokens.

In [11]:
if tokenizer is None:
    print("No trained tokenizer found, skipping tensor/mask example.")
else:
    ids, mask = tokenizer.render_conversation(toy_conversation)
    batch_tensor = torch.tensor([ids], dtype=torch.long)
    mask_tensor = torch.tensor([mask], dtype=torch.int8)

    inputs = batch_tensor[:, :-1].to(dtype=torch.int32).contiguous()
    targets = batch_tensor[:, 1:].to(dtype=torch.int64).contiguous()

    mask_targets = mask_tensor[:, 1:]
    targets[mask_targets == 0] = -1

    print("Shapes:")
    print_kv([
        ("batch_tensor", tuple(batch_tensor.shape)),
        ("inputs", tuple(inputs.shape)),
        ("targets", tuple(targets.shape)),
    ])
    print()
    print(f"{'pos':>3}  {'input token':<22} -> {'target token / ignored'}")
    print("-" * 70)
    for pos in range(inputs.shape[1]):
        input_text = tokenizer.decode([int(inputs[0, pos])]).replace("\n", "\\n")
        target_id = int(targets[0, pos])
        if target_id == -1:
            target_text = "IGNORE (-1)"
        else:
            target_text = repr(tokenizer.decode([target_id]).replace("\n", "\\n"))
        print(f"{pos:>3}  {repr(input_text):<22} -> {target_text}")

Shapes:
batch_tensor               (1, 19)
inputs                     (1, 18)
targets                    (1, 18)

pos  input token            -> target token / ignored
----------------------------------------------------------------------
  0  '<|bos|>'              -> IGNORE (-1)
  1  '<|user_start|>'       -> IGNORE (-1)
  2  'What'                 -> IGNORE (-1)
  3  ' is'                  -> IGNORE (-1)
  4  ' '                    -> IGNORE (-1)
  5  '2'                    -> IGNORE (-1)
  6  '+'                    -> IGNORE (-1)
  7  '2'                    -> IGNORE (-1)
  8  '?'                    -> IGNORE (-1)
  9  '<|user_end|>'         -> IGNORE (-1)
 10  '<|assistant_start|>'  -> '2'
 11  '2'                    -> '+'
 12  '+'                    -> '2'
 13  '2'                    -> ' is'
 14  ' is'                  -> ' '
 15  ' '                    -> '4'
 16  '4'                    -> '.'
 17  '.'                    -> '<|assistant_end|>'


## Step 7. Tool Outputs Are Context, Not Supervised Targets

GSM8K and SpellingBee can represent assistant content as parts, not just one string.

For example:

```python
{"type": "python", "text": "2+2"}
{"type": "python_output", "text": "4"}
```

The assistant's tool call is supervised, but the Python output is not supervised. Why? Because at inference time the tool output should come from the tool, not from the model hallucinating it.

In `render_conversation`, `python_output` tokens get `mask=0`.

In [12]:
tool_conversation = {
    "messages": [
        {"role": "user", "content": "Use Python to compute 2+2."},
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": "I will calculate it.\n"},
                {"type": "python", "text": "2+2"},
                {"type": "python_output", "text": "4"},
                {"type": "text", "text": "\nThe answer is 4."},
            ],
        },
    ]
}

if tokenizer is None:
    print("No trained tokenizer found, skipping tool-output example.")
else:
    tool_ids, tool_mask = tokenizer.render_conversation(tool_conversation)
    for i, (token_id, mask_value) in enumerate(zip(tool_ids, tool_mask)):
        decoded = tokenizer.decode([token_id]).replace("\n", "\\n")
        if "python" in decoded or decoded in {"2", "+", "4"} or mask_value == 0:
            print(f"{i:>2} mask={mask_value} id={token_id:<6} {decoded!r}")
    print()
    print("Look for <|output_start|> ... <|output_end|>: those tokens and output text are mask=0.")

 0 mask=0 id=32759  '<|bos|>'
 1 mask=0 id=32760  '<|user_start|>'
 2 mask=0 id=5436   'Use'
 3 mask=0 id=9045   ' Python'
 4 mask=0 id=283    ' to'
 5 mask=0 id=17651  ' compute'
 6 mask=0 id=32     ' '
 7 mask=0 id=50     '2'
 8 mask=0 id=43     '+'
 9 mask=0 id=50     '2'
10 mask=0 id=46     '.'
11 mask=0 id=32761  '<|user_end|>'
12 mask=0 id=32762  '<|assistant_start|>'
18 mask=1 id=32764  '<|python_start|>'
19 mask=1 id=50     '2'
20 mask=1 id=43     '+'
21 mask=1 id=50     '2'
22 mask=1 id=32765  '<|python_end|>'
23 mask=0 id=32766  '<|output_start|>'
24 mask=0 id=52     '4'
25 mask=0 id=32767  '<|output_end|>'
31 mask=1 id=52     '4'

Look for <|output_start|> ... <|output_end|>: those tokens and output text are mask=0.


## Step 8. SFT Packs Whole Conversations Into Rows

SFT examples have different lengths. `chat_sft.py` packs full conversations into rows of length `max_seq_len + 1`.

It tries to fit whole conversations, not crop arbitrary chunks. If no conversation fits the remaining space, it pads with BOS tokens and mask `0`.

That gives the same final shape as base training:

```text
inputs:  (device_batch_size, max_seq_len)
targets: (device_batch_size, max_seq_len)
```

but targets have many `-1` positions because user/padding/tool-output tokens are ignored.

In [13]:
def pack_bestfit_toy(conversations, row_capacity, bos_token):
    buffer = list(conversations)
    rows = []
    mask_rows = []
    while buffer:
        row = []
        mask_row = []
        while len(row) < row_capacity:
            remaining = row_capacity - len(row)
            best_idx = -1
            best_len = 0
            for i, (conv_ids, _) in enumerate(buffer):
                if len(conv_ids) <= remaining and len(conv_ids) > best_len:
                    best_idx = i
                    best_len = len(conv_ids)
            if best_idx == -1:
                pad = row_capacity - len(row)
                row.extend([bos_token] * pad)
                mask_row.extend([0] * pad)
                break
            conv_ids, conv_mask = buffer.pop(best_idx)
            row.extend(conv_ids)
            mask_row.extend(conv_mask)
        rows.append(row)
        mask_rows.append(mask_row)
    return rows, mask_rows

if tokenizer is None:
    print("No trained tokenizer found, skipping packing example.")
else:
    short_conversations = [
        {"messages": [{"role": "user", "content": "A?"}, {"role": "assistant", "content": "B."}]},
        {"messages": [{"role": "user", "content": "What is 1+1?"}, {"role": "assistant", "content": "2."}]},
        {"messages": [{"role": "user", "content": "Say hi."}, {"role": "assistant", "content": "Hi."}]},
    ]
    rendered = [tokenizer.render_conversation(c) for c in short_conversations]
    bos = tokenizer.get_bos_token_id()
    rows, mask_rows = pack_bestfit_toy(rendered, row_capacity=48, bos_token=bos)

    for row_idx, (row, mask_row) in enumerate(zip(rows, mask_rows)):
        supervised = sum(mask_row)
        ignored = len(mask_row) - supervised
        print(f"row {row_idx}: length={len(row)}, supervised_tokens={supervised}, ignored_tokens={ignored}")
        decoded_preview = tokenizer.decode(row[:40]).replace("\n", "\\n")
        print(f"  preview: {truncate(decoded_preview, 180)}")

row 0: length=48, supervised_tokens=9, ignored_tokens=39
  preview: <|bos|><|user_start|>What is 1+1?<|user_end|><|assistant_start|>2.<|assistant_end|><|bos|><|user_start|>Say hi.<|user_end|><|assistant_start|>Hi.<|assistant_end|><|bos|><|user_s...


## Step 9. The Training Step Is The Same Shape As Base Training

Once SFT has built `x` and `y`, the model does not know whether the batch came from parquet pretraining text or chat examples.

The loop still does:

```text
loss = model(x, y)
loss.backward()
optimizer.step()
```

The SFT-specific behavior is mostly in the data preparation:

```text
conversation -> ids + mask -> shifted inputs/targets -> targets masked with -1
```

That is a helpful simplification: SFT does not require a different model architecture.

In [14]:
if tokenizer is None:
    print("No trained tokenizer found, skipping final tensor check.")
else:
    ids, mask = tokenizer.render_conversation(toy_conversation)
    batch_tensor = torch.tensor([ids], dtype=torch.long)
    mask_tensor = torch.tensor([mask], dtype=torch.int8)
    x = batch_tensor[:, :-1].to(dtype=torch.int32).contiguous()
    y = batch_tensor[:, 1:].to(dtype=torch.int64).contiguous()
    y[mask_tensor[:, 1:] == 0] = -1

    print("This is the object that reaches gpt.py during SFT:")
    print_kv([
        ("x dtype", x.dtype),
        ("x shape", tuple(x.shape)),
        ("y dtype", y.dtype),
        ("y shape", tuple(y.shape)),
        ("ignored targets", int((y == -1).sum().item())),
        ("supervised targets", int((y != -1).sum().item())),
    ])
    print()
    print("Real call shape:")
    print("  loss = model(x, y)")

This is the object that reaches gpt.py during SFT:
x dtype                    torch.int32
x shape                    (1, 18)
y dtype                    torch.int64
y shape                    (1, 18)
ignored targets            10
supervised targets         8

Real call shape:
  loss = model(x, y)


## Step 10. Validation BPB And ChatCORE During SFT

SFT evaluates two different things:

- `Validation bpb`: loss/compression on held-out SFT conversations.
- `ChatCORE`: benchmark score across ARC, MMLU, GSM8K, HumanEval, SpellingBee.

BPB tells you whether the model predicts the SFT validation data well.

ChatCORE tells you whether the chat/task behavior is improving.

In [15]:
def show_matching_lines(path, patterns, limit=80):
    path = Path(path)
    if not path.exists():
        print(f"Missing log file: {path}")
        return
    regexes = [re.compile(pattern) for pattern in patterns]
    shown = 0
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line_no, line in enumerate(f, 1):
            if any(regex.search(line) for regex in regexes):
                print(f"{line_no:>5}: {line.rstrip()}")
                shown += 1
                if shown >= limit:
                    print(f"... stopped after {limit} matches")
                    break

sft_log = artifact_root / "sft_retry.log"
patterns = [
    r"Training mixture",
    r"Step 0?\d+ \| Validation bpb",
    r"ChatCORE",
    r"Saved model parameters.*chatsft_checkpoints",
    r"Minimum validation bpb",
    r"ARC-Easy accuracy",
    r"MMLU accuracy",
    r"GSM8K accuracy",
    r"HumanEval accuracy",
    r"SpellingBee accuracy",
]
show_matching_lines(sft_log, patterns, limit=80)

  115: Training mixture: 1,071,759 rows (MMLU x3, GSM8K x4)
  116: Step 00000 | Validation bpb: 0.4430
  317: Step 00200 | Validation bpb: 0.3055
  450: Step 00200 | ChatCORE: 0.3302 | ChatCORE_cat: 0.2993
  651: Step 00400 | Validation bpb: 0.2804
  763: Step 00400 | ChatCORE: 0.3515 | ChatCORE_cat: 0.3140
  847: Step 00483 | Validation bpb: 0.2734
  959: Step 00483 | ChatCORE: 0.3618 | ChatCORE_cat: 0.3207
  967: 2026-05-28 01:39:09,465 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /workspace/nanochat-cache/chatsft_checkpoints/d24/model_000483.pt
  972: Minimum validation bpb: 0.2734
 1144: ARC-Easy accuracy: 61.99%
 1148: MMLU accuracy: 36.00%
 2478: GSM8K accuracy: 8.64%
 2653: HumanEval accuracy: 12.20%
 2920: SpellingBee accuracy: 99.22%


## Step 11. SFT Saves A New Checkpoint Folder

At the end, SFT saves into:

```text
$NANOCHAT_BASE_DIR/chatsft_checkpoints/d24/
```

For your run this produced:

```text
model_000483.pt          -> chat model weights
meta_000483.json         -> architecture/config metadata
optim_000483_rank*.pt    -> optimizer shards for resuming SFT
```

For chat inference you normally use the SFT checkpoint, not the base checkpoint.

In [16]:
sft_ckpt_dir = artifact_root / "chatsft_checkpoints" / "d24"
if sft_ckpt_dir.exists():
    for path in sorted(sft_ckpt_dir.glob("*")):
        size = path.stat().st_size
        print(f"{path.name:<28} {size / 1024 / 1024:>8.1f} MB")
else:
    print(f"No SFT checkpoint folder found at {sft_ckpt_dir}")

meta_000483.json                  0.0 MB
model_000483.pt                4032.1 MB
optim_000483_rank0.pt           684.2 MB
optim_000483_rank1.pt           684.2 MB
optim_000483_rank2.pt           684.2 MB
optim_000483_rank3.pt           684.2 MB
optim_000483_rank4.pt           684.2 MB
optim_000483_rank5.pt           684.2 MB
optim_000483_rank6.pt           684.2 MB
optim_000483_rank7.pt           684.2 MB


## Mental Model

SFT is not a new model type. It is the same GPT model trained on different examples with a different target mask.

```text
Base training:
  raw text tokens -> predict every next normal token

SFT:
  conversation tokens -> predict assistant tokens only
```

That single mask difference is a big deal. It tells the model: the user prompt is input context, and the assistant response is what you should learn to produce.